In [8]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import optimize
from scipy import stats
import statsmodels.formula.api as smf 

# Also - set up plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline
plt.style.use('fivethirtyeight')

## Homework 2
### Marking criteria

- Read questions carefully and do what the question asks
- Point values are included with each question
- E.g

```
  #- Put the "Region" column of the data frame into a new variable "regions".
#- 2 marks / 100 (total 2 so far)
regions = ...
# Show the result
regions
```
- There are 100 points overall
- There are some plots to make.  Look carefully at what the question asks for


Now display a **horizontal bar plot** of the ethnicities. When you have done this correctly, you should see the ethnicities listed on the left of the vertical axis, with the least common ethnicity first. The horizontal axis corresponds to percent. You should see coloured bars going from left to right, with the longest bar at the bottom, corresponding to the percentage of "White British".

Hint: Use tab completion to investigate the methods contained in west_midlands_sorted.plot for a plausible plot type to try.

- Label plots
- Do not 'hard code' answers
  - i.e. don't just set a variable to be equal to an answer you got from somewhere else
  - Use code to calculate the answers
  - Make the code robust (e.g. if you need the last item use `my_array[-1]` or `my_array[len(my_array)-1]` not a constant, e.g. `my_array[37]`
<br>

- '...' generally means you need to put something here
- store your answers in the variables provided
  - e.g. `last_item = ...`
  - store your answer in `last_item`, e.g. `last_item = my_array[-1]`
  - do not change the names of the answer variables


There are a few questions that ask for a multiple-choice answer, e.g.:

Looking at this histogram, how likely is it, that the proportion that we observe for the number of "White British" people stopped, could have come about under the assumptions of our ideal model?

Assign either 1, 2, 3, 4, or 5 to the name likely_no_bias below.

1) Reasonably likely.
1) Fairly unlikely, but not unlikely enough to be surprising.
1) Unlikely, but we still cannot be confident the model is incorrect.
1) Highly unlikely, we have strong evidence the model is incorrect.
1) The observed result is impossible given the model assuming no difference.

The answer cell looks like this:

```
#- 5 marks / 100 (total 55 so far)
likely_no_bias = ...
```

Set likely_no_bias = (the number of the answer you choose 1-5)


### Questions about homework 2?

### Binomial logistic regression (also called binary logistic regression or logistic regression) - 
### Part 1 - changing two-alternative data into data suitable for linear regression - i.e. Make it linear!

There is more detail about this topic in the textbook here: https://uob-ds.github.io/cfd2022/more-regression/logistic_regression.html

- Binomial (two-number) regression is used when we are predicting something that can only take two values
- Examples:
    - Dead/alive
    - Success/failure
    - Disease/no disease
    - Presence/Absence
<br>

Sounds like count data.  What is different?
#### The _predictor_ is not categorical.  Here the predictor is a continous value

- Examples:
    - Dead/alive - Predicting 5 year cancer suvival based on tumour size at presentation
    - Success/failure - Predicting loan defaults (failure to repay) based on debt to income ratio
    - Disease/no disease - Predicting lung cancer from daily number of cigarettes smoked
    - Presence/Absence - Predicting psychosis based on amount of marijuana typically consumed in a month

We can't use normal linear regression techniques for this situation.  Why not?

#### Linear regression wants:
- A linear relationship - binary outcomes don't fall on a line, they are 0/1 or probability of 0/1 - see below
- Predicting unbounded numbers - binary outcomes are only 0/1 (or probability values from 0 to 1)
    - Linear regeression assumes we are predicting unbounded continuous values (not just 0/1 or a probability between 0 and 1)
    - Linear regression predicted values are unbounded and can be negative or positive
      - we might predict a negative probability - that is not possible

### To reiterate: Why aren't  two-value situations like this?

- Our data are 0/1, but what we **predict** is:
- the **_probability_** of 0 and the **_probability_** of 1
    - e.g. The probability of disease vs no disease at each level of our continuous predictor value
    - the probability of psychosis for each level of marijuana use
    
### And so?
- If predicted values are probabilities...
    - Values are not linear (see below)
    - Values are bounded between zero and 1
        - no negative numbers
        - no numbers larger than one

####  Instead, simple linear regression predicts:
- Values on a line.
- Predicted values from an _**unlimited**_ set of numbers, not just 2 possibilities
  - i.e. Predicted values are not _bounded_
  - The values are not restricted to being between 0 and 1

### Why don't predicted probabilites of 0/1 (e.g. predicted probability of psychosis) fall on a straight line?

<img src=sigmoid_function.png width=400 align=right>

- The typical function that describes probabilities in a two-value situation is a "logistic function"
- This is an S-shaped curve that increases slowly at the beginning, faster in the middle and slowly again at the end
<br>

- A typical example is learning a new sport.  Think of training time on the x axis and ability on the y-axis.
- At the very beginning progress is slow.
- Once you start to get the hang of the basics you improve more rapidly
- To get to expert level, though, is much slower.
    - The final steps to reach a high level take a long time
<br> 


<img src=SlamDunk.png width=600 align=right>

- Our example below will be dunking a basketball and height
  - When people are sufficiently short, nobody can dunk a basketball
  - Once a certain height is reached a few people (good jumpers) can do it
  - When people are even taller, the change is rapid and many more people start to be able to dunk
  - When people are really tall (like most pros) everyone can dunk
 <br>
 
 #### This is not a linear function and it is bounded (by 'can't dunk ever' and 'can always dunk')


### Using a straight line in this situation can have serious negative consequences
- predict impossible values
- make very bad predictions at the beginning or the end of the function
  - e.g. when will you be an expert
    - Assume 'practice' is on the x axis

### But we only know about linear regression - what to do?


- Call Mick (https://www.youtube.com/watch?v=oqMl5CRoFdk)
    - Oh dear - A reference to what my children call "daddy music" -- sorry.  Couldn't resist.
      - **"You don't always get what you want...you just might find, you get what you need"**
  <br>

    - A collection of 70-year old geriatrics (at that time of the video)
    - Compare to the younger version (https://www.youtube.com/watch?v=Ef9QnZVpVd8)
    - Well, we all get older....

<img src=RollingStones.png align=right width=500>

#### Our strategy: Don't have what you want?  (a straight line situation)?
#### Get what you need:
- Convert the problem into a linear regression problem
<br>
 
- Convert probabilities into another value that is:
  - linear
  - unbounded
  - continuous
- Use linear techniques to predict your converted value
  - Use our normal linear regression techniques on our converted values
- Reconvert the predicted value back to a probability value
<br>

## How?

### First step - Convert from 0-1 to 0-unbounded

- Probability values must be between 0 and 1 - This is our first problem


#### Solution - part 1

- Note that any probability in a two-choice situation has its opposite
    - The probability of success determines the probability of failure
    - The two options must add up to 1
    - If the probability of success is 80%, the probability of failure must be 20%
 <br>
 
 We can use this and turn any probability in an **_odds ratio_**
 
- An odds ratio asks: How much more likely is success than failure?
- 80% / 20% means that success is 4 times more likely than failure
- This fraction can take on any number from 0 to infinity
<br>
  
Odds ration equation: 
 
$$\frac{p(success)}{p(failure)}$$
 
 - or, since 
 
 $$p(failure) = 1 - p(success)$$
 
- odds ratio equation: 
 
$$\frac{p(success)}{(1 - p(success))}$$

Notice that, as we've said, an odds ratio is an unbounded number between 0 and positive infinity

#### Now we are half way there, but only halfway...
#### There is still a bound at 0

### Review where we are up to

Possible outcomes are:
- success is more likely than failure
- failure is more likely than success
- even odds
<br>

- If even odds

$$p(success) = p(failure)$$  
$$\frac{p(success)}{p(failure)} = 1$$  

- If success more likely
$$\frac{p(success) = big\ number}{p(failure) = small\ number}$$  
    - Result > 1
    - range: 1 to infinity
    
- If failure more likely
$$\frac{p(success) = small\ number}{p(failure) = big\ number}$$    
    - Result between 1 and 0
    
    
#### That hardly seems fair.  
- When success is more likely we've got all the possible numbers > 1
- When failure is more likely our numbers are smashed between 0 and 1

Plus we are still bounded on the lower side by zero.

### Solution step 2 -  Take the log

You may remember that:

$$10^1 = 10$$
$$log_{10}{10} = 1$$
<br>
  
$$10^2 = 100$$
$$log_{10}(100) = 2$$  
<br>
  
$$10^3 = 1000$$
$$log_{10}(1000) = 3$$
<br>
  
General rule:
$$ x^y = z$$
$$ log_{x}(z) = y$$

You may also remember that
$$0.1 = 10^{-1}$$
$$0.01 = 10^{-2}$$
$$0.001 = 10^{-3}$$

So after taking the log, the numbers between 0 and 1 are converted to negative numbers between 0 and negative infinity.  Now we have numbers between negative infinity and positive infinity, with even odds at the zero point.

Our normal linear regression techinques can predict a log(odds_ratio) and then we can convert that to a probability to see how closely the probability matches the actual values.

## Summary

If we start from predicted probabilities:

p(failure) = 1 - p(success) 

$$odds\ ratio = \frac{p(success)}{(1 - p(success))}$$

$$log\ odds = log(odds\ ratio)$$

OR (equivalently)

$$log\ odds = log(\frac{p(success)}{p(failure)})$$

#### This converts probabilities of success/failure that are 0-1 to continuous numbers from -infinity to +infinity
#### We can also start by predicting log odds 
#### When we have our answer we can convert back to probabilities


#### You might see taking step 1 and step 2 together labelled -- "taking the log odds"


All righty then!  Now I can use my usual regression tools (we will need to develop a new error measure next time)

#### Preparing for an analysis

- What this means for an analysis:
  - We will need a function that converts probabilities into log odds.
  - We will need a function that converts log odds into probabilities.

### Math review - recall (see also above) that log and $e$ are opposites

In [7]:
# for example

import numpy as np

# ln10 = natural log of 10
ln10 = np.log(10)
print(f"natural log of 10 is {ln10:.4f}")
# note that log calculates the natural logarithm - log10 calculates the log using a base of 10

natural log of 10 is 2.3026


In [16]:
test = np.log10(10)
print(f"log to the base 10 of 10 = {test:.4f}")

log to the base 10 of 10 = 1.0000


In [12]:
# to reverse the log process we can take e to the power of x

should_be_10 = np.exp(ln10)
print(f"This value should be 10: {should_be_10:.4f}")

This value should be 10: 10.0000


### log and $e$ are opposites

**if**

$$log(x) = y$$

**then:**

$$e^y = x$$

### Converting back and forth between p(success) and log odds
#### Converting probabilities into log odds

In [20]:
# let's do an example first
prob_success = 0.9
prob_fail = 1-prob_success
odds_ratio = prob_success/prob_fail
print(f"The odds ratio is: {odds_ratio:.4f}")
log_odds = np.log(odds_ratio)
print(f"log odds: {log_odds:.4f}")

The odds ratio is: 9.0000
log odds: 2.1972


In [ ]:
# write your function here

### Converting log(odds) into p(success)
#### Do this in two steps
#### 1) log(odds) $\rightarrow$ odds_ratio
#### 2) odds_ratio $\rightarrow$ p(success)
<br>

#### Step 1: Convert log(odds) to the odds ratio

In [19]:
# using our knowledge that e and log are opposites
odds_ratio = np.exp(log_odds)
print(f"odds_ratio = {odds_ratio:.4f}")

odds_ratio = 9.0000


In [ ]:
# write your function here

####  Step 2: Converting an odds ratio to p(success)

### We want p(success) and we have the odds_ratio - How do we convert back?

#### We need to do some algebra:

$$ odds\_ratio = \frac{p(success)}{(1-p(success))}$$
Multiply both sides by $(1-p(success))$
$$ odds\_ratio * (1-p(success)) = p(success) $$
Expanding the left side multiplying by the odds_ratio:
$$ odds\_ratio  - (odds\_ratio * p(success)) = p(success)$$
Moving $odds\_ratio * p(success)$ to the right side:
$$ odds\_ratio = p(success) + odds\_ratio*p(success)$$
Factoring
$$ odds\_ratio = p(success) * (1 + odds\_ratio)$$
Divide both sides by $(1 + odds\_ratio)$
$$\frac{odds\_ratio}{(1 + odds\_ratio)} = p(success)$$

In [22]:
# write a function to convert a log(odds_ratio) value to p(success)
# convert log(odds_ratio) to odds_ratio
# convert odds_ratio to p(success)
# you can use a function you've already written for a step

In [28]:
# test
odds_ratio = 1
p_success = odds_ratio/(1+odds_ratio)

print(f"If the odds ratio is {odds_ratio}, the probability of success is: {p_success:.2f} and the p_failure is {1-p_success:.2f}")

If the odds ratio is 1, the probability of success is: 0.50 and the p_failure is 0.50


### When we pick up again I will assume you have gone through this exercise to write these functions and understand converting back and forth
### The rest resembles very closely what we have already done with linear regression
### The main thing that we still will need to do is find an error measure that works with probability values